# Walkthrough

The aim of this notebook is to do a complete tour of the functionalities and possibilities of `Analytical-Satsky`.

## 1. Imports
We start by importing standard libraries and the needed functions.

In [1]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import EarthLocation

from analytical_satsky import compute_occupancy_fraction
from analytical_satsky import load_constellation, list_constellations
from analytical_satsky import MultiShellObs


%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


## 2. Shells

Many orbital shells are predefined within `Analytical-Satsky`. They are regrouped in one DataFrame for each major constellation.

To know the available constellations use `list_constellations`


In [2]:
print(list_constellations())


['guowang', 'leo', 'oneweb', 'qianfan', 'starlink_filing1', 'starlink_filing2', 'starlink_march25', 'starlink_scaled40000']


You can then use `load_constellation`, that can take as argument several strings. They must be part of the list above, but it is case insensitive.

In [3]:
guowang_and_leo = load_constellation("guowang", "leo")
guowang_and_leo


,i,h,n
0,85.0,590,480
1,50.0,600,2000
2,55.0,508,3600
3,30.0,1145,1728
4,40.0,1145,1728
5,50.0,1145,1728
6,60.0,1145,1728
7,51.9,630,1156
8,42.0,610,1296
9,33.0,590,784


You can also create your own constellation configuration, as long as you respect the Dataframe construction: `i` is the orbit inclination in degrees, `h` the orbit altitude in km and `n` the number of satellites populating the shell.

In [4]:
my_const = pd.DataFrame(columns=['i', 'h', 'n'], data=[[45, 500, 1000], [45, 1000, 200]])
my_const

,i,h,n
0,45,500,1000
1,45,1000,200


## 3. Observation specs

Now you need to define the observing condition: which telescope ? pointing where ? For how long ?.

Note that the model does not depend on the time of the observation, and uses the local hour angle instead of the right ascension.

The Observatory is an `astropy.coordinates.EarthLocation` object, so you may use predefined locations.

In [5]:
skalow = EarthLocation.of_site("SKA-Low")

# instead you can also create your own location:
# my_observatory = EarthLocation.from_geodetic(lon=116.5*u.deg, lat=-40*u.deg, height=0*u.m)

# Take an target near the zenith of SKA-Low at the beginning of the observation
target_dec = np.array([-30]) * u.deg
target_lha = np.array([0]) * u.deg

Lfov = 10*u.deg # roughly the Main Beam size at 100 MHz
texp = 3600*u.s # Observe for 1 hour

The model also works on a batch of astronomical target. 

Give 1d arrays as `target_dec` and `target_lha` to define a grid of coordinates where the model will be evaluated.

In [6]:
skalow = EarthLocation.of_site("SKA-Low")

# Take an grid of targets in every direction (including below the horizon)
ndec, nra = 90*10, 360*5
target_dec = np.linspace(-90, 90, ndec, endpoint=False)*u.deg
target_lha = np.linspace(0, 360, nra, endpoint=False)*u.deg

Lfov = 10*u.deg # roughly the Main Beam size at 100 MHz
texp = 3600*u.s # Observe for 1 hour

## 4. Using the model

### 4.1 Deterministic properties

You then call `MultiShellObs`. Deterministic results are stored as properties.

The main one is `total_nsats`, which is the aggregated number of expected satellites from all shells.

To get the repartition of the satellites per shell, see `nsats_per_shell`.


In [7]:
# Build the model
multi_shell_model = MultiShellObs(skalow, guowang_and_leo, target_dec, target_lha, Lfov, texp)
total_dens = multi_shell_model.total_nsats
print("Shapes of target_dec:", target_dec.shape)
print("Shapes of target_lha:", target_lha.shape)
print("Shapes of total_dens:", total_dens.shape)

/Users/nicolascerardi/miniconda3/lib/python3.12/site-packages/astropy/units/quantity.py:653: RuntimeWarning: invalid value encountered in arcsin
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


Shapes of target_dec: (900,)
Shapes of target_lha: (1800,)
Shapes of total_dens: (1800, 900)


The model uses lazy caching of its properties. When you access total_dens once, you can access all other parent properties without compute.

This includes `nsats_per_shell` but also properties of individual shells, listed in the `shell_models` property.

In [8]:
# Access the number of satellites for the last shell, for all pointing directions
print(multi_shell_model.nsats_per_shell[-1].shape)

# Get the array of distances to the first shell, for all pointing directions
print(multi_shell_model.shell_models[0].d.shape)

# Accessing these quantities should take 0s

(1800, 900)
(1800, 900)


### 4.2. Stochastic predictions

From the deterministic number of satellites, the model allows to sample catalogues of fake satellites from each shell.

This in turn allows to estimate the fraction of time with satellites occupying the Main Beam.

The method `sample_passes` allows to sample a catalogue from the model, while the function `compute_occupancy_fraction` estimates the statistic $ f_{\rm N_{sat} \geq 1} $ (see the paper for details).

In [9]:
skalow = EarthLocation.of_site("SKA-Low")
Lfov = 10.0 *u.deg
texp = 3600*u.s
target_decs = np.arange(10, -80, -5)*u.deg
target_lha = np.array([0.])*u.deg
ntimestep = int(texp.to(u.s).value * 5)

fractions_q16 = np.zeros_like(target_decs.value)
fractions_q5   = np.zeros_like(target_decs.value)
fractions_q84 = np.zeros_like(target_decs.value)

for i in range(len(target_decs)):
    target_dec = target_decs[[i]]
    multi_shell_obs = MultiShellObs(skalow, guowang_and_leo, target_dec, target_lha, Lfov, texp)
    all_inits, all_ts = multi_shell_obs.sample_passes(nstat=100)
    fractions = compute_occupancy_fraction(texp, ntimestep, all_inits, all_ts)